In [ ]:
from analysisManager import stage

In [ ]:
load = 1e3 # 1kN load
length = 2 # 2m long cantilever
youngsModulus = 200e8 # for steel
density = 7850 # for steel

def rectangularCantileverDeflection(depth,width,**kwargs):
	I = depth**3 * width / 12
	deflection = load * length**3 / (4 * youngsModulus * I)
	return {'deflection':deflection}

def getMass(depth,width,**kwargs):
	volume = depth * width * length
	mass = volume * density
	return {'mass':mass}

deflectionStage = stage(rectangularCantileverDeflection, inputs = ['depth','width'],name = 'Cantilever deflection')
massStage = stage(getMass,inputs = ['depth','width'],name = 'Mass')

stages = [deflectionStage,massStage] 

In [ ]:
from analysisManager import basicSampler

sampler = basicSampler(
	depth = (0.3,0.75),
	width = (0.1,0.3)
)

In [ ]:
from analysisManager import searchRoutine, optimiseRoutine

searcher = searchRoutine(sampler,method = 'sobol',density = 6) # you can search in different patterns

from numpy import inf
valueFunction = lambda row: [row.get('mass',inf), row.get('deflection',inf)] # this is the function that will be optimised. It takes a row of the runManager.df DataFrame and returns a list of values to be minimised. In this case, we want to minimise both mass and deflection.
optimiser = optimiseRoutine(sampler = sampler,targetFunction = valueFunction)

In [ ]:
from analysisManager import runManager

runner = runManager(stages = stages, routines = [searcher,optimiser], name='Example',verbose = False)
runner.runRoutines()

In [ ]:
candidates = searcher.getResultPopulation(target = 'deflection',nLen = 10) # get the 10 best candidates from the search routine, based on the deflection value. This can be used as the initial population for the optimisation routine.
newOptimiser = optimiseRoutine(sampler = sampler,targetFunction = valueFunction,initialPopulation = candidates) # add these to a new optimisation routine, which will use these candidates as the initial population for the optimisation. This is useful if you want to do a more focused optimisation after a broad search.
runner.runRoutine(newOptimiser)

In [ ]:
optimisation_configuration = {
	'n_gen': 40,
	'pop_size': 30,
	'sampling':candidates,
	'n_offsprings':20
}
runner.runRoutines(**optimisation_configuration) 

Running routines can also take additional keyword arguments, which will be passed to the routines. In this case, we are passing the optimisation configuration to the optimiser routine.
Note:
All routines receive the same keyword arguments, so run the routines separately if you want more granular control.
Like...

In [ ]:
runner.runRoutine(searcher)
runner.runRoutine(optimiser,**optimisation_configuration)

All of the returned results are stored in dataframes.
If results are suitable to be stored in a dataframe, return it in the result dictionary

Each design point has its own directory which holds its inputs in an "inputs.json" file and a folder for each stage.

In [ ]:
# All of the returned results are stored in dataframes
# Each routine stores its own results (routine.df)
# The Run Manager stores results from all of its routines in a single dataframe (runManager.df)
# You can plot each one independently, or create a combined plot.

completePlot = runner.df.plot.scatter(x='mass',y='deflection',c='r',label='All results')
optimiserPlot = optimiser.df.plot.scatter(x='mass',y='deflection',c='b',label='Optimisation results')

You can swap out stages from the same run manager if you want

In [ ]:
def I_beamDeflection(depth,width,**kwargs):
	I = (width * depth**3)/12 - ((width-0.1) * (depth-0.1)**3)/12
	deflection = load * length**3 / (4 * youngsModulus * I)
	return {'deflection':deflection}